In [18]:
import pandas as pd
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler

from geoai.utils_geo.raster_ops import RasterOperations
from geoai.utils_ds.dataframe_ops import DataFrameOperations


raster_ops = RasterOperations()
df_ops = DataFrameOperations()

In [6]:
RASTER_PATH = r"raster_files\QC_2018_2023.tif"
N_BANDS = 5
BAND_NAMES = ["BLUE", "GREEN", "RED", "NIR", "SWIR"]

In [7]:
df_bands = []
array = raster_ops.raster_to_array(RASTER_PATH)
raster_dimension = raster_ops.get_raster_dimensions(RASTER_PATH)


for band_index, band_name in zip(range(N_BANDS), BAND_NAMES):
    flat = raster_ops.flatten_array(array, band_index)
    df = df_ops.convert_to_df(flat, band_name)
    df = df.loc[~(df == 0).all(axis=1)]  # remove rows if all of its column is zero
    df_bands.append(df)
final_df_per_bands = pd.concat(df_bands, axis=1).astype('float64')
final_df_per_bands.head()


,BLUE,GREEN,RED,NIR,SWIR
0,1337.500000,1697.000000,1712.000000,2578.000000,2172.5
1,1374.666626,1663.333374,1638.000000,2679.366699,2185.5
2,1370.500000,1576.000000,1592.800049,2183.333252,2202.5
3,1318.000000,1446.666626,1404.000000,1894.833374,2202.5
4,994.333313,1136.666626,1119.000000,1732.000000,1922.0


In [8]:
with open("trained_models/kmeans_4.pkl", "rb") as file:
    kmeans_4 = pickle.load(file)

with open("trained_models/kmeans_5.pkl", "rb") as file:
    kmeans_5 = pickle.load(file)

final_df_per_bands_cluster4 = final_df_per_bands.copy()
final_df_per_bands_cluster5 = final_df_per_bands.copy()

final_df_per_bands_cluster4["CLUSTER_4"] = kmeans_4.predict(final_df_per_bands_cluster4)
final_df_per_bands_cluster5["CLUSTER_5"] = kmeans_5.predict(final_df_per_bands_cluster5)

In [9]:
raster_ops.column_to_raster(
    "raster_files/CLUSTER_4.tif",
    final_df_per_bands_cluster4,
    "CLUSTER_4",
    raster_dimension,
    "float32",
)

raster_ops.column_to_raster(
    "raster_files/CLUSTER_5.tif",
    final_df_per_bands_cluster5,
    "CLUSTER_5",
    raster_dimension,
    "float32",
)

In [10]:
final_df_per_bands = raster_ops.indices_binary_category(final_df_per_bands) 

In [11]:
with open("trained_models/cat.pkl", "rb") as file:
    cat = pickle.load(file)
cat

Pipeline(steps=[('pipeline_4',
                 FeatureUnion(transformer_list=[('pipeline_1',
                                                 ColumnTransformer(transformers=[('categorical_transformer_1',
                                                                                  Pipeline(steps=[('one_hot_transformer',
                                                                                                   OneHotEncoder(dtype=<class 'int'>,
                                                                                                                 sparse_output=False))]),
                                                                                  ['NDVI_binary']),
                                                                                 ('categorical_transformer_2',
                                                                                  Pipeline(steps=[('ordinal_transformer',
                                                                                                   OrdinalEncoder(categories=[['low_...
                                                                 ('pca_transformer',
                                                                  PCA(n_components=7))]))])),
                ('select_important_features',
                 FunctionTransformer(func=<function ModelOperations.select_important_features at 0x00000208660DCCC0>)),
                ('print_shape',
                 FunctionTransformer(func=<function ModelOperations.print_shape at 0x0000020866BC22A0>)),
                ('classifier',
                 <catboost.core.CatBoostClassifier object at 0x00000208660E1B10>)])

In [12]:
final_df_per_bands["PREDICTED_LC"] = cat.predict(final_df_per_bands)
final_df_per_bands.head(5)

Shape before feature selection: (158224, 57)
Shape after feature selection: (158224, 21)


,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,NDVI_categorized,NDVI_binary,PREDICTED_LC
0,1337.500000,1697.000000,1712.000000,2578.000000,2172.5,0.201865,-0.085359,0.000359,medium_veg,non_veg,2
1,1374.666626,1663.333374,1638.000000,2679.366699,2185.5,0.241204,-0.101517,0.000354,medium_veg,non_veg,2
2,1370.500000,1576.000000,1592.800049,2183.333252,2202.5,0.156386,0.004370,0.000271,low_veg,non_veg,2
3,1318.000000,1446.666626,1404.000000,1894.833374,2202.5,0.148790,0.075089,0.000231,low_veg,non_veg,2
4,994.333313,1136.666626,1119.000000,1732.000000,1922.0,0.215012,0.051998,0.000428,medium_veg,non_veg,2


In [13]:
raster_ops.column_to_raster(
    "raster_files/PREDICTED_LC_CAT.tif",
    final_df_per_bands,
    "PREDICTED_LC",
    raster_dimension,
    "float32",
)

In [14]:
with open("trained_models/simple_pipeline.pkl", "rb") as file:
    simple_pipe = pickle.load(file)
simple_pipe

Pipeline(steps=[('min_max_scaler', MinMaxScaler()),
                ('lr', LogisticRegression(max_iter=10000, random_state=1))])

In [15]:
final_df_per_bands["PREDICTED_LC_SIMPLE"] = simple_pipe.predict(final_df_per_bands[["BLUE","GREEN","RED"]])
final_df_per_bands.head(5)

raster_ops.column_to_raster(
    "raster_files/PREDICTED_LC_SIMPLE.tif",
    final_df_per_bands,
    "PREDICTED_LC_SIMPLE",
    raster_dimension,
    "float32",
)

In [20]:
rgbnirswir = final_df_per_bands[["RED", "GREEN", "BLUE", "NIR", "SWIR"]]
rgbnirswir

,RED,GREEN,BLUE,NIR,SWIR
0,1712.000000,1697.000000,1337.500000,2578.000000,2172.5
1,1638.000000,1663.333374,1374.666626,2679.366699,2185.5
2,1592.800049,1576.000000,1370.500000,2183.333252,2202.5
3,1404.000000,1446.666626,1318.000000,1894.833374,2202.5
4,1119.000000,1136.666626,994.333313,1732.000000,1922.0
...,...,...,...,...,...
158219,873.000000,971.333313,1012.000000,2283.333252,2902.5
158220,961.000000,1049.333374,1049.500000,2343.000000,2677.0
158221,975.000000,1049.333374,1046.000000,2310.000000,2677.0
158222,722.000000,771.000000,765.333313,1962.500000,2168.5


In [22]:
scaler = MinMaxScaler()
rgbnirswir_scaled = scaler.fit_transform(rgbnirswir).astype(np.float32)
rgbnirswir_scaled_torch = torch.tensor(rgbnirswir_scaled)

In [23]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(5, 64)  
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 4)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x


model = Net()
model.load_state_dict(torch.load('best_model_state_dict.pth'))
model.eval()

# Predict
with torch.no_grad():
    outputs = model(rgbnirswir_scaled_torch)
    _, predicted = torch.max(outputs, 1)
    predictions = predicted.cpu().numpy()


C:\Users\Reginald\AppData\Local\Temp\ipykernel_13136\3229173280.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model_state_dict.

In [26]:
final_df_per_bands['PREDICTED_LC_NN'] = predictions
final_df_per_bands

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,NDVI_categorized,NDVI_binary,PREDICTED_LC,PREDICTED_LC_SIMPLE,PREDICTED_LC_NN
0,1337.500000,1697.000000,1712.000000,2578.000000,2172.5,0.201865,-0.085359,0.000359,medium_veg,non_veg,2,0,2
1,1374.666626,1663.333374,1638.000000,2679.366699,2185.5,0.241204,-0.101517,0.000354,medium_veg,non_veg,2,0,1
2,1370.500000,1576.000000,1592.800049,2183.333252,2202.5,0.156386,0.004370,0.000271,low_veg,non_veg,2,0,1
3,1318.000000,1446.666626,1404.000000,1894.833374,2202.5,0.148790,0.075089,0.000231,low_veg,non_veg,2,2,0
4,994.333313,1136.666626,1119.000000,1732.000000,1922.0,0.215012,0.051998,0.000428,medium_veg,non_veg,2,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
158219,1012.000000,971.333313,873.000000,2283.333252,2902.5,0.446826,0.119396,0.000550,medium_veg,non_veg,1,2,1
158220,1049.500000,1049.333374,961.000000,2343.000000,2677.0,0.418281,0.066534,0.000526,medium_veg,non_veg,1,2,1
158221,1046.000000,1049.333374,975.000000,2310.000000,2677.0,0.406393,0.073591,0.000523,medium_veg,non_veg,1,2,1
158222,765.333313,771.000000,722.000000,1962.500000,2168.5,0.462097,0.049867,0.000796,medium_veg,non_veg,3,1,1


In [27]:
raster_ops.column_to_raster(
    "raster_files/PREDICTED_LC_NN.tif",
    final_df_per_bands,
    "PREDICTED_LC_NN",
    raster_dimension,
    "float32",
)

END